In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ── Sources 
silver = spark.table("iran_israel_capstone_project.silver.daily_market_clean")
event_dim = spark.table("iran_israel_capstone_project.silver.event_dim")

# ── HIGH / CRITICAL event dates 
hc_events = (
    event_dim
    .filter(F.col("severity").isin("HIGH", "CRITICAL"))
    .select("event_date", "event_id", "severity")
    .distinct()
)
hc_dates = hc_events.select(F.col("event_date").alias("hc_date"))

# ── Base daily flow table (non-null FII rows only)
daily = (
    silver
    .filter(F.col("fii_net_cr").isNotNull())
    .select(
        "trade_date", "fii_net_cr", "fii_gross_buy_cr", "fii_gross_sell_cr",
        "dii_net_cr", "usdinr_close", "usdinr_daily_change_pct",
        "nifty_close", "nifty_daily_return_pct",
        "event_id", "event_type", "severity"
    )
)

# ── KPI 1: Flag HIGH/CRITICAL event days & FII selling 
# The silver table already has severity from the left-join;
# a row has severity IN (HIGH, CRITICAL) when it's an event day.
daily = daily.withColumn(
    "is_hc_event_day",
    F.col("severity").isin("HIGH", "CRITICAL")
).withColumn(
    "fii_is_selling",
    F.col("fii_net_cr") < 0
)

# ── KPI 4 prep: Row-number for proximity calculation 
# Assign row number by trade_date so we can measure "trading days"
w_date = Window.orderBy("trade_date")
daily = daily.withColumn("row_num", F.row_number().over(w_date))

# For each trading day, check if it is within 5 TRADING days of any
# HIGH/CRITICAL event. We assign row numbers to HC event dates too.
hc_trade_dates = (
    daily
    .filter(F.col("is_hc_event_day"))
    .select(F.col("trade_date").alias("hc_trade_date"), F.col("row_num").alias("hc_row_num"))
    .distinct()
)

# Cross join daily with HC dates, filter within 5 trading days
proximity = (
    daily.select("trade_date", "row_num")
    .crossJoin(hc_trade_dates)
    .filter(F.abs(F.col("row_num") - F.col("hc_row_num")) <= 5)
    .select("trade_date")
    .distinct()
    .withColumn("within_5td_of_hc", F.lit(True))
)

daily = daily.join(proximity, on="trade_date", how="left").fillna({"within_5td_of_hc": False})

# ── KPI 4: Rank by FII outflow (most negative first)
w_outflow = Window.orderBy(F.col("fii_net_cr").asc())
daily = daily.withColumn("fii_outflow_rank", F.row_number().over(w_outflow))

# ── Write gold table 
gold_fii = daily.select(
    "trade_date", "fii_net_cr", "fii_gross_buy_cr", "fii_gross_sell_cr",
    "dii_net_cr", "usdinr_close", "usdinr_daily_change_pct",
    "nifty_close", "nifty_daily_return_pct",
    "event_id", "event_type", "severity",
    "is_hc_event_day", "fii_is_selling", "within_5td_of_hc", "fii_outflow_rank"
)

gold_fii.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "iran_israel_capstone_project.gold.gold_fii_flow_analysis"
)

print(f" Gold FII flow table written: {gold_fii.count()} rows")


# KPI SUMMARY


# KPI 1: FII sell rate on HIGH+ event days
hc_days = gold_fii.filter(F.col("is_hc_event_day"))
total_hc = hc_days.count()
fii_selling_hc = hc_days.filter(F.col("fii_is_selling")).count()
sell_rate = round(fii_selling_hc / total_hc * 100, 1) if total_hc > 0 else 0
print(f"\n── KPI 1: FII Sell Rate on HIGH/CRITICAL Days ──")
print(f"   {fii_selling_hc}/{total_hc} days FII was net seller = {sell_rate}% (target >= 70%)")

# KPI 2: DII counter-flow (data unavailable)
dii_avail = gold_fii.filter(F.col("dii_net_cr").isNotNull()).count()
print(f"\n── KPI 2: DII Counter-Flow ──")
if dii_avail == 0:
    print("    DII data unavailable (100% null) — cannot compute cushion rate")
else:
    heavy_fii_sell = gold_fii.filter(F.col("fii_net_cr") < -5000)
    dii_cushion = heavy_fii_sell.filter(F.col("dii_net_cr") > 0).count()
    cushion_rate = round(dii_cushion / heavy_fii_sell.count() * 100, 1) if heavy_fii_sell.count() > 0 else 0
    print(f"   {dii_cushion}/{heavy_fii_sell.count()} days DII absorbed = {cushion_rate}%")

# KPI 3: Rupee correlation
from pyspark.sql import Row
corr_df = (
    gold_fii
    .filter(F.col("fii_net_cr").isNotNull() & F.col("usdinr_daily_change_pct").isNotNull())
)
r_value = corr_df.stat.corr("fii_net_cr", "usdinr_daily_change_pct")
print(f"\n── KPI 3: FII–Rupee Correlation ──")
print(f"   Pearson r = {round(r_value, 4)} (negative expected: FII selling → rupee weakening)")
print(f"   Observation: {'Confirmed negative ' if r_value < 0 else 'Positive — unexpected '}")

# KPI 4: Top 5 FII outflow days — proximity to HC events
top5 = gold_fii.filter(F.col("fii_outflow_rank") <= 5).orderBy("fii_outflow_rank")
overlap_count = top5.filter(F.col("within_5td_of_hc")).count()
print(f"\n── KPI 4: Top-5 FII Outflow Days near HC Events ──")
print(f"   Overlap with HIGH/CRITICAL (±5 trading days): {overlap_count}/5")
display(top5.select("fii_outflow_rank", "trade_date", "fii_net_cr", "event_id", "severity", "is_hc_event_day", "within_5td_of_hc"))

In [0]:
# ── Spot-check KPI 1: manually count FII selling on HC days ──
from pyspark.sql import functions as F

gold = spark.table("iran_israel_capstone_project.gold.gold_fii_flow_analysis")

# KPI 1 verification
hc = gold.filter(F.col("is_hc_event_day"))
print(f"HC event days: {hc.count()}")
print(f"FII selling days: {hc.filter(F.col('fii_is_selling')).count()}")
print(f"FII buying days:  {hc.filter(~F.col('fii_is_selling')).count()}")
print()

# Show HC event days with FII flow details
display(
    hc.select("trade_date", "fii_net_cr", "event_id", "severity", "fii_is_selling")
    .orderBy("trade_date")
)

# KPI 3 verification: correlation from the gold table
print(f"\nGold table Pearson r (fii_net_cr, usdinr_daily_change_pct):")
print(f"  {round(gold.filter(F.col('usdinr_daily_change_pct').isNotNull()).stat.corr('fii_net_cr', 'usdinr_daily_change_pct'), 4)}")

# KPI 4 verification: top 5 outflow days
print(f"\nTop-5 outflow days:")
display(
    gold.orderBy("fii_net_cr")
    .select("trade_date", "fii_net_cr", "within_5td_of_hc", "event_id", "severity")
    .limit(5)
)